# TFT - Weekly Revenue Forecast (8-week horizon)

Ноутбук повторяет структуру статьи и минимально адаптирует её под weekly `City`-уровень проекта.

## 1. Загрузка и подготовка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import List

from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer
from darts.utils.likelihood_models import QuantileRegression
from darts.explainability import TFTExplainer

import torch
import torchmetrics

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OrdinalEncoder
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.lr_monitor import LearningRateMonitor

## 2. Формирование датафреймов target и covariates

In [ ]:
df = pd.read_csv("data/sales.csv", sep=";")
df["date"] = pd.to_datetime(df["Week"])
df = (
    df.groupby(["City", "date"], as_index=False)["revenue"]
    .sum()
    .sort_values(["City", "date"])
    .reset_index(drop=True)
)

city_frames = []
for city, city_frame in df.groupby("City", sort=True):
    full_weeks = pd.date_range(city_frame["date"].min(), city_frame["date"].max(), freq="W-MON")
    city_grid = pd.DataFrame({"date": full_weeks})
    city_grid["City"] = city
    city_grid = city_grid.merge(city_frame, on=["City", "date"], how="left")
    city_grid["is_missing"] = city_grid["revenue"].isna().astype(int)
    city_grid["revenue"] = city_grid["revenue"].fillna(0.0)
    city_frames.append(city_grid)

df = pd.concat(city_frames, ignore_index=True).sort_values(["City", "date"]).reset_index(drop=True)
print(f"Series: {df['City'].nunique()}, Date range: {df['date'].min()} - {df['date'].max()}")
print(f"Total rows: {len(df)}")

In [ ]:
iso_calendar = df["date"].dt.isocalendar()
df["week_of_year"] = iso_calendar.week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year
df["is_holiday_week"] = 0

shifted = df.groupby("City")["revenue"].shift(1)
df["lag_1"] = shifted
df["lag_2"] = df.groupby("City")["revenue"].shift(2)
df["lag_4"] = df.groupby("City")["revenue"].shift(4)
df["lag_8"] = df.groupby("City")["revenue"].shift(8)
df["rolling_4"] = shifted.groupby(df["City"]).rolling(4).mean().reset_index(level=0, drop=True)
df["rolling_8"] = shifted.groupby(df["City"]).rolling(8).mean().reset_index(level=0, drop=True)
df["rolling_12"] = shifted.groupby(df["City"]).rolling(12).mean().reset_index(level=0, drop=True)

df_target = df[["date", "City", "revenue"]].copy()
df_past_covs = df[
    ["date", "City", "lag_1", "lag_2", "lag_4", "lag_8", "rolling_4", "rolling_8", "rolling_12"]
].copy()
df_future_covs = df[["date", "City", "week_of_year", "month", "quarter", "year", "is_holiday_week"]].copy()

display(df_target.head())
display(df_past_covs.head())
display(df_future_covs.head())

## 3. Формирование TimeSeries

In [ ]:
# заполним на следующем шаге

## 4. Нормализация

In [ ]:
# заполним на следующем шаге

## 5. Разделение на train / val / test

In [ ]:
# заполним на следующем шаге

## 6. Создание и обучение TFT

In [ ]:
# заполним на следующем шаге

## 7. Выполнение прогнозов

In [ ]:
# заполним на следующем шаге

## 8. Оценка качества и визуализация

In [ ]:
# заполним на следующем шаге

## 9. Интерпретация результатов

In [ ]:
# заполним на следующем шаге